In [1]:
import pandas as pd
import os
from glob import glob
import warnings
import re

In [2]:
warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl")

# 设置文件夹路径
folder_path = "raw_data/Raw_data_weight"  # 替换为你的本地路径
file_list = glob(os.path.join(folder_path, "*.xlsx"))

def extract_cleaned_data_from_file(file_path):
    try:
        xls = pd.ExcelFile(file_path)
        sheet_names = [s for s in xls.sheet_names if s.startswith("Sheet")]
        if not sheet_names:
            print(f"⚠️ 跳过：无 Sheet 页 -> {file_path}")
            return None

        frames = []
        for sheet in sheet_names:
            try:
                df = pd.read_excel(xls, sheet_name=sheet, header=None)

                # 提取元信息（容错处理）
                frequency = df.iloc[4, 2] if pd.notna(df.iloc[4, 2]) else df.iloc[4, 0]
                product   = df.iloc[5, 2] if pd.notna(df.iloc[5, 2]) else df.iloc[5, 0]
                flow      = df.iloc[6, 2] if pd.notna(df.iloc[6, 2]) else df.iloc[6, 0]
                indicators= df.iloc[7, 2] if pd.notna(df.iloc[7, 2]) else df.iloc[7, 0]

                df_data = pd.read_excel(xls, sheet_name=sheet, header=9)

                # 结构校验
                if df_data.shape[0] < 2 or df_data.shape[1] < 2:
                    print(f"⚠️ 跳过 sheet（结构异常）: {sheet} in {file_path}")
                    continue

                # 删除无效第二行（常为冒号）
                df_data = df_data.drop(index=0).reset_index(drop=True)

                # 标准化列名
                df_data.columns.values[0:2] = ['country', 'date']

                # 添加元信息
                df_data["Sheet"] = sheet
                df_data["SourceFile"] = os.path.basename(file_path)
                df_data["Frequency"] = frequency
                df_data["PRODUCT"] = product
                df_data["FLOW"] = flow
                df_data["INDICATORS"] = indicators

                frames.append(df_data)

            except Exception as e:
                print(f"❌ Sheet 跳过: {sheet} in {file_path}\n{e}")
                continue

        if frames:
            return pd.concat(frames, ignore_index=True)
        else:
            print(f"⚠️ 文件跳过（无有效 sheet 数据）: {file_path}")
            return None

    except Exception as e:
        print(f"❌ 文件读取失败: {file_path}\n{e}")
        return None

# 批量读取所有文件，过滤无效项
df_list = [extract_cleaned_data_from_file(fp) for fp in file_list]
df_list = [df for df in df_list if df is not None]

# 合并所有数据
if df_list:
    all_data = pd.concat(df_list, ignore_index=True)

    # 宽转长：国家列 → partner/value
    metadata_cols = ['country', 'date', 'Sheet', 'SourceFile', 'Frequency', 'PRODUCT', 'FLOW', 'INDICATORS']
    value_cols = [col for col in all_data.columns if col not in metadata_cols]

    df_long = all_data.melt(
        id_vars=metadata_cols,
        value_vars=value_cols,
        var_name='partner',
        value_name='weight'
    )

    # 替换冒号为空值，强制转换为 float
    df_long["weight"] = pd.to_numeric(df_long["weight"].replace({":": None}), errors="coerce")

    # ✅ 保存结果（可选）
    # df_long.to_csv("combined_cleaned_long.csv", index=False)
    # df_long.to_excel("combined_cleaned_long.xlsx", index=False)

    # ✅ 预览前几行
    print(df_long.head())
else:
    print("❗ 所有文件均未成功读取或无有效数据。")


   country     date    Sheet      SourceFile Frequency  \
0  Austria  2022-01  Sheet 1  data_2022.xlsx   Monthly   
1  Austria  2022-02  Sheet 1  data_2022.xlsx   Monthly   
2  Austria  2022-03  Sheet 1  data_2022.xlsx   Monthly   
3  Austria  2022-04  Sheet 1  data_2022.xlsx   Monthly   
4  Austria  2022-05  Sheet 1  data_2022.xlsx   Monthly   

                          PRODUCT    FLOW         INDICATORS  partner  weight  
0  New pneumatic tyres, of rubber  IMPORT  QUANTITY_IN_100KG  Andorra     NaN  
1  New pneumatic tyres, of rubber  IMPORT  QUANTITY_IN_100KG  Andorra     NaN  
2  New pneumatic tyres, of rubber  IMPORT  QUANTITY_IN_100KG  Andorra     NaN  
3  New pneumatic tyres, of rubber  IMPORT  QUANTITY_IN_100KG  Andorra     NaN  
4  New pneumatic tyres, of rubber  IMPORT  QUANTITY_IN_100KG  Andorra     NaN  


In [3]:
df_long.info() 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16737231 entries, 0 to 16737230
Data columns (total 10 columns):
 #   Column      Dtype  
---  ------      -----  
 0   country     object 
 1   date        object 
 2   Sheet       object 
 3   SourceFile  object 
 4   Frequency   object 
 5   PRODUCT     object 
 6   FLOW        object 
 7   INDICATORS  object 
 8   partner     object 
 9   weight      float64
dtypes: float64(1), object(9)
memory usage: 1.2+ GB


In [4]:
df_long.columns

Index(['country', 'date', 'Sheet', 'SourceFile', 'Frequency', 'PRODUCT',
       'FLOW', 'INDICATORS', 'partner', 'weight'],
      dtype='object')

In [5]:
df_long

,country,date,Sheet,SourceFile,Frequency,PRODUCT,FLOW,INDICATORS,partner,weight
0,Austria,2022-01,Sheet 1,data_2022.xlsx,Monthly,"New pneumatic tyres, of rubber",IMPORT,QUANTITY_IN_100KG,Andorra,NaN
1,Austria,2022-02,Sheet 1,data_2022.xlsx,Monthly,"New pneumatic tyres, of rubber",IMPORT,QUANTITY_IN_100KG,Andorra,NaN
2,Austria,2022-03,Sheet 1,data_2022.xlsx,Monthly,"New pneumatic tyres, of rubber",IMPORT,QUANTITY_IN_100KG,Andorra,NaN
3,Austria,2022-04,Sheet 1,data_2022.xlsx,Monthly,"New pneumatic tyres, of rubber",IMPORT,QUANTITY_IN_100KG,Andorra,NaN
4,Austria,2022-05,Sheet 1,data_2022.xlsx,Monthly,"New pneumatic tyres, of rubber",IMPORT,QUANTITY_IN_100KG,Andorra,NaN
...,...,...,...,...,...,...,...,...,...,...
16737226,Slovakia,2021-11,Sheet 7,data_2021.xlsx,Monthly,"New pneumatic tyres, of rubber, of a kind used...",IMPORT,QUANTITY_IN_100KG,Zimbabwe,NaN
16737227,Slovakia,2021-12,Sheet 7,data_2021.xlsx,Monthly,"New pneumatic tyres, of rubber, of a kind used...",IMPORT,QUANTITY_IN_100KG,Zimbabwe,NaN
16737228,NaN,NaN,Sheet 7,data_2021.xlsx,Monthly,"New pneumatic tyres, of rubber, of a kind used...",IMPORT,QUANTITY_IN_100KG,Zimbabwe,NaN
16737229,Special value,NaN,Sheet 7,data_2021.xlsx,Monthly,"New pneumatic tyres, of rubber, of a kind used...",IMPORT,QUANTITY_IN_100KG,Zimbabwe,NaN


In [6]:
df_long = df_long[['country', 'date', 'PRODUCT', 'INDICATORS', 'partner', 'weight']]
df_long

,country,date,PRODUCT,INDICATORS,partner,weight
0,Austria,2022-01,"New pneumatic tyres, of rubber",QUANTITY_IN_100KG,Andorra,NaN
1,Austria,2022-02,"New pneumatic tyres, of rubber",QUANTITY_IN_100KG,Andorra,NaN
2,Austria,2022-03,"New pneumatic tyres, of rubber",QUANTITY_IN_100KG,Andorra,NaN
3,Austria,2022-04,"New pneumatic tyres, of rubber",QUANTITY_IN_100KG,Andorra,NaN
4,Austria,2022-05,"New pneumatic tyres, of rubber",QUANTITY_IN_100KG,Andorra,NaN
...,...,...,...,...,...,...
16737226,Slovakia,2021-11,"New pneumatic tyres, of rubber, of a kind used...",QUANTITY_IN_100KG,Zimbabwe,NaN
16737227,Slovakia,2021-12,"New pneumatic tyres, of rubber, of a kind used...",QUANTITY_IN_100KG,Zimbabwe,NaN
16737228,NaN,NaN,"New pneumatic tyres, of rubber, of a kind used...",QUANTITY_IN_100KG,Zimbabwe,NaN
16737229,Special value,NaN,"New pneumatic tyres, of rubber, of a kind used...",QUANTITY_IN_100KG,Zimbabwe,NaN


In [7]:
df_long['weight'] = df_long['weight']*100

/var/folders/_6/_zms2dyd4l5d2rlgnk6pq6cr0000gp/T/ipykernel_4755/2300936854.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_long['weight'] = df_long['weight']*100


In [8]:
df_long["country"].unique()

array(['Austria', "Belgium (incl. Luxembourg 'LU' -> 1998)", 'Bulgaria',
       'Cyprus', 'Czechia',
       "Germany (incl. German Democratic Republic 'DD' from 1991)",
       'Denmark', 'Estonia',
       "Spain (incl. Canary Islands 'XB' from 1997)", 'Finland',
       "France (incl. Saint Barthélemy 'BL' -> 2012; incl. French Guiana 'GF', Guadeloupe 'GP', Martinique 'MQ', Réunion 'RE' from 1997; incl. Mayotte 'YT' from 2014)",
       'United Kingdom', 'Greece', 'Croatia', 'Hungary', 'Ireland (Eire)',
       "Italy (incl. San Marino 'SM' -> 1993)", 'Lithuania', 'Luxembourg',
       'Latvia', 'Malta', 'Netherlands', 'Poland', 'Portugal', 'Romania',
       'Sweden', 'Slovenia', 'Slovakia', nan, 'Special value', ':'],
      dtype=object)

In [9]:
def clean_label(value):
    if pd.isna(value):
        return value
    value = re.sub(r"\s*\(.*", "", value)                       # 括号内容
    value = re.sub(r"\s*from\s+\d{4}.*?\)$", "", value)         # from 1994 -> 2000)
    value = re.sub(r"\s*->\s*\d{4}\)?$", "", value)             # -> 1994)
    return value.strip()

In [10]:
# 删除括号及其内容：Belgium (incl. Luxembourg 'LU' -> 1998) → Belgium
df_long["country"] = df_long["country"].apply(clean_label)

df_long["partner"] = df_long["partner"].apply(clean_label)

# date
df_long["date"] = pd.to_datetime(df_long["date"], errors="coerce", format="%Y-%m")
df_long["year"] = df_long["date"].dt.year
df_long["month"] = df_long["date"].dt.month

/var/folders/_6/_zms2dyd4l5d2rlgnk6pq6cr0000gp/T/ipykernel_4755/4282884942.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_long["country"] = df_long["country"].apply(clean_label)
/var/folders/_6/_zms2dyd4l5d2rlgnk6pq6cr0000gp/T/ipykernel_4755/4282884942.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_long["partner"] = df_long["partner"].apply(clean_label)
/var/folders/_6/_zms2dyd4l5d2rlgnk6pq6cr0000gp/T/ipykernel_4755/4282884942.py:7: SettingWithCopyWarning: 
A value is trying to be set on a c

In [11]:
df_long["date"].unique()

<DatetimeArray>
['2022-01-01 00:00:00', '2022-02-01 00:00:00', '2022-03-01 00:00:00',
 '2022-04-01 00:00:00', '2022-05-01 00:00:00', '2022-06-01 00:00:00',
 '2022-07-01 00:00:00', '2022-08-01 00:00:00', '2022-09-01 00:00:00',
 '2022-10-01 00:00:00',
 ...
 '2021-03-01 00:00:00', '2021-04-01 00:00:00', '2021-05-01 00:00:00',
 '2021-06-01 00:00:00', '2021-07-01 00:00:00', '2021-08-01 00:00:00',
 '2021-09-01 00:00:00', '2021-10-01 00:00:00', '2021-11-01 00:00:00',
 '2021-12-01 00:00:00']
Length: 313, dtype: datetime64[ns]

In [12]:
df_long["PRODUCT"].unique()

array(['New pneumatic tyres, of rubber',
       'New pneumatic tyres, of rubber, of a kind used for motor cars, incl. station wagons and racing cars',
       'New pneumatic tyres, of rubber, of a kind used for buses and lorries (excl. tyres with lug, corner or similar treads)',
       'Pneumatic tyres, new, of rubber, of a kind used for buses or lorries, with a load index of <= 121',
       'Pneumatic tyres, new, of rubber, of a kind used for buses or lorries, with a load index of > 121',
       'New pneumatic tyres, of rubber, of a kind used on agricultural or forestry vehicles and machines',
       'New pneumatic tyres, of rubber, of a kind used on construction, mining or industrial handling vehicles and machines'],
      dtype=object)

In [13]:
hs_code_map = {
    "New pneumatic tyres, of rubber": "4011",
    "New pneumatic tyres, of rubber, of a kind used for motor cars, incl. station wagons and racing cars": "401110",
    "New pneumatic tyres, of rubber, of a kind used for buses and lorries (excl. tyres with lug, corner or similar treads)": "401120",
    'Pneumatic tyres, new, of rubber, of a kind used for buses or lorries, with a load index of <= 121': '40112010',
    'Pneumatic tyres, new, of rubber, of a kind used for buses or lorries, with a load index of > 121': '40112090',
    "New pneumatic tyres, of rubber, of a kind used on agricultural or forestry vehicles and machines": "401170",
    "New pneumatic tyres, of rubber, of a kind used on construction, mining or industrial handling vehicles and machines": "401180",
}

product_map = {
    "New pneumatic tyres, of rubber": "All",
    "New pneumatic tyres, of rubber, of a kind used for motor cars, incl. station wagons and racing cars": "PCR",
    "New pneumatic tyres, of rubber, of a kind used for buses and lorries (excl. tyres with lug, corner or similar treads)": "LT&TBR",
    'Pneumatic tyres, new, of rubber, of a kind used for buses or lorries, with a load index of <= 121': 'LT',
    'Pneumatic tyres, new, of rubber, of a kind used for buses or lorries, with a load index of > 121': 'TBR',
    "New pneumatic tyres, of rubber, of a kind used on agricultural or forestry vehicles and machines": "ARG & FOR",
    "New pneumatic tyres, of rubber, of a kind used on construction, mining or industrial handling vehicles and machines": "GC & Mining & MH",
}

product_map_categoty = {
    "PCR": "PCR",
    "LT": 'PCR',
    "TBR": "TBR",
    "LT&TBR": "LT&TBR",
    "ARG & FOR": "Specialty",
    "GC & Mining & MH": "Specialty",
    'All': 'All'
}

In [14]:
df_long["HS Code"] = df_long["PRODUCT"].map(hs_code_map)
df_long["Category"] = df_long["PRODUCT"].map(product_map)
df_long["General_Category"] = df_long["Category"].map(product_map_categoty)

/var/folders/_6/_zms2dyd4l5d2rlgnk6pq6cr0000gp/T/ipykernel_4755/3871727594.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_long["HS Code"] = df_long["PRODUCT"].map(hs_code_map)


In [15]:
df_long["HS Code"].unique()

array(['4011', '401110', '401120', '40112010', '40112090', '401170',
       '401180'], dtype=object)

df_long.dropna(subset=['country'], inplace=True)

In [17]:
df_long_filter = df_long[(df_long['country'] != 'Special value')&(df_long['country'] != ':')&(df_long['country'] != 'European Union - 27 countries')]

In [18]:
df_long_filter['partner'].unique()

array(['Andorra', 'United Arab Emirates', 'Afghanistan',
       'Antigua and Barbuda', 'Anguilla', 'Albania', 'Armenia',
       'Netherlands Antilles', 'Angola', 'Antarctica', 'Argentina',
       'American Samoa', 'Austria', 'Australia', 'Aruba', 'Azerbaijan',
       'Bosnia and Herzegovina', 'Barbados', 'Bangladesh', 'Belgium',
       'Burkina Faso', 'Bulgaria', 'Bahrain', 'Burundi', 'Benin',
       'Saint Barthélemy', 'Bermuda', 'Brunei Darussalam',
       'Bolivia, Plurinational State of',
       'Bonaire, Sint Eustatius and Saba', 'Brazil', 'Bahamas', 'Bhutan',
       'Bouvet Island', 'Botswana', 'Belarus', 'Belize', 'Canada',
       'Cocos Islands', 'Congo, Democratic Republic of',
       'Central African Republic', 'Congo', 'Switzerland',
       'Côte d’Ivoire', 'Cook Islands', 'Chile', 'Cameroon', 'China',
       'Colombia', 'Costa Rica', 'Serbia and Montenegro', 'Cuba',
       'Cabo Verde', 'Curaçao', 'Christmas Island', 'Cyprus', 'Czechia',
       'German Democratic Republic',

In [19]:
exclude_values = [
    'All countries of the world',
    'Extra-EU27', 
    'Intra-EU27',
    'Stores and provisions',
    'Stores and provisions within the framework of intra-Union trade',
    'Stores and provisions within the framework of extra-Union trade',
    'Countries and territories not specified',
    'Countries and territories not specified within the framework of intra-Union trade',
    'Countries and territories not specified within the framework of extra-Union trade',
    'Countries and territories not specified for commercial or military reasons',
    'Countries and territories not specified for commercial or military reasons in the framework of intra-Union trade',
    'Countries and territories not specified for commercial or military reasons in the framework of extra-Union trade'
]

df_long_filter = df_long_filter[~df_long_filter['partner'].isin(exclude_values)]

In [20]:
df_long_filter.info()

<class 'pandas.core.frame.DataFrame'>
Index: 16075010 entries, 0 to 16737228
Data columns (total 11 columns):
 #   Column            Dtype         
---  ------            -----         
 0   country           object        
 1   date              datetime64[ns]
 2   PRODUCT           object        
 3   INDICATORS        object        
 4   partner           object        
 5   weight            float64       
 6   year              float64       
 7   month             float64       
 8   HS Code           object        
 9   Category          object        
 10  General_Category  object        
dtypes: datetime64[ns](1), float64(3), object(7)
memory usage: 1.4+ GB


In [21]:
# 仅删除 value 列中为 NaN 的行
df_long_filter_drop = df_long_filter.dropna(subset=['weight'])

In [22]:
df_long_filter_drop

,country,date,PRODUCT,INDICATORS,partner,weight,year,month,HS Code,Category,General_Category
96,Spain,2022-01-01,"New pneumatic tyres, of rubber",QUANTITY_IN_100KG,Andorra,460.0,2022.0,1.0,4011,All,All
97,Spain,2022-02-01,"New pneumatic tyres, of rubber",QUANTITY_IN_100KG,Andorra,48.0,2022.0,2.0,4011,All,All
98,Spain,2022-03-01,"New pneumatic tyres, of rubber",QUANTITY_IN_100KG,Andorra,300.0,2022.0,3.0,4011,All,All
99,Spain,2022-04-01,"New pneumatic tyres, of rubber",QUANTITY_IN_100KG,Andorra,5.0,2022.0,4.0,4011,All,All
100,Spain,2022-05-01,"New pneumatic tyres, of rubber",QUANTITY_IN_100KG,Andorra,96.0,2022.0,5.0,4011,All,All
...,...,...,...,...,...,...,...,...,...,...,...
16725843,United Kingdom,2000-07-01,"New pneumatic tyres, of rubber, of a kind used...",QUANTITY_IN_100KG,Zimbabwe,4800.0,2000.0,7.0,401110,PCR,PCR
16732623,United Kingdom,2001-07-01,"New pneumatic tyres, of rubber",QUANTITY_IN_100KG,Zimbabwe,4400.0,2001.0,7.0,4011,All,All
16732628,United Kingdom,2001-12-01,"New pneumatic tyres, of rubber",QUANTITY_IN_100KG,Zimbabwe,5000.0,2001.0,12.0,4011,All,All
16732962,United Kingdom,2001-07-01,"New pneumatic tyres, of rubber, of a kind used...",QUANTITY_IN_100KG,Zimbabwe,4400.0,2001.0,7.0,401110,PCR,PCR


In [23]:
df_long_filter_drop.to_csv('raw_data/tire_product_Weight_PCR_TBR_SP.csv',index=False)

In [24]:
df_long_filter_drop[(df_long_filter_drop['date'] == "2022-07-01")&(df_long_filter_drop['HS Code']=='401110')]['weight'].sum()

671123828.0